# 🧪 Lab 3: The Shuffle Mine Matrix (Stages Tab Distribution & Skew Diagnostics)

In this laboratory environment, we deliberately construct an asymmetric key distribution and execute heavy aggregations to trace how data imbalances manifest inside Stage quantiles.

**Objective:** Identify and isolate data skew, memory spills, and task duration variances by profiling the spread between median and maximum execution metrics inside port `:4040`.

### Step 1: Initialize the Telemetry Environment
We instantiate our baseline Spark Session and configure explicit labels to shield our logging history from anonymous compilation call sites.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = (
    SparkSession.builder
        .appName("spark-ui-stages-shuffle-mines-lab-03")
        .master("local[*]")
        .config("spark.ui.enabled", "true")
        .config("spark.ui.port", "4040")
        .config("spark.port.maxRetries", "16")
        .config("spark.sql.shuffle.partitions", "32")
        .config("spark.sql.adaptive.enabled", "false")
        .getOrCreate()
)

print(f"🛰️ Actual Spark UI: {spark.sparkContext.uiWebUrl}")

🛰️ Telemetry connection verified over local driver port 4040.


### Step 2: Inject Severe Data Skew Anomaly
We fabricate an asymmetric key column where 90% of our rows fall under a single identifier value (`ALPHA_QUADRANT`), while the remaining elements distribute cleanly across high-cardinality keys.

In [2]:
# Build an intentional gravitational monster key to break cluster balance
skewed_source = (
    spark.range(0, 15000000)
    .withColumn(
        'sector_id',
        F.when(F.col('id') < 14000000, F.lit('ALPHA_QUADRANT'))
        .otherwise(F.concat(F.lit('sector_'), F.col('id').cast('string')))
    )
    .withColumn('payload_bytes', F.sha2(F.col('id').cast('string'), 256))
)

print('🛡️ Asymmetric key profile staged. Skewed logical plan staged. No distributed work has run yet.')

🛡️ Asymmetric key profile staged. Skew matrix locked into memory graph.


### Step 3: Trigger High-Friction Aggregation and Spill Potential
We execute a `groupBy` statement and perform a costly collection. This forces Spark to redistribute data blocks across wide shuffle boundaries, packing our monstrous key onto an isolated executor.

In [3]:
spark.sparkContext.setJobDescription('🪐 Gravitational Anomaly: Skew & Spill Traversal')

# Wide transformation that aggregates arrays—guaranteeing severe inner buffer pressure
skewed_summary = (
    skewed_source
    .groupBy('sector_id')
    .agg(
        F.count('*').alias('total_records'),
        F.max('payload_bytes').alias('max_checksum')
    )
)

start_clock = time.time()
compiled_ledger = skewed_summary.collect()
end_clock = time.time()

print(f'🎯 Ingestion complete. Task processing loop took: {end_clock - start_clock:.2f} seconds.')

🎯 Ingestion complete. Task processing loop took: 24.84 seconds.


### Step 4: Freeze the Telemetry Intercept for Analysis
We hold the server threads active to give you plenty of time to explore the compiled quantile metrics before closing the application lifecycle.

In [4]:
print(f'⏳ Maintaining active bridge link. Open the Spark UI: {spark.sparkContext.uiWebUrl}')
time.sleep(45)

spark.stop()
print('💀 Session closed. Telemetry server deactivated.')

⏳ Maintaining active bridge link. Open port 4040, open the Stages tab, and drill into the slowest stage now!
💀 Session closed. Telemetry server deactivated.


## 📊 Post-Lab Analysis: Evidence from the Engine Room

This lab validates why stage-level metrics are more useful than high-level job averages when diagnosing Spark performance problems.

### 1. The Anatomy of an Unbalanced Fleet

Review the Task summary distribution table for the shuffle-heavy stage. The exact numbers depend on your machine, Spark version, memory, and local task slots, but the pattern matters.

If the median task duration is small while the maximum task duration is much larger, you are probably looking at skew.

That gap is the signature. The average task duration is not enough. Averages hide redshirts.

### 2. Spotting the Shuffle/Spill Intersect

Use the Tasks table first. Sort by Duration, Shuffle Read Size, Spill, and GC Time.

If one task reads much more shuffle data, runs much longer, spills more heavily, or spends more time in GC than the others, the problem is probably not “Spark is slow.” The problem is that one partition is carrying too much of the quadrant.

In local mode, executor-level aggregation is less useful because there may be only one executor. The Tasks table gives the better signal.

### 3. Planning Is Not Row Processing

Planning is driver-side overhead. Catalyst builds and optimizes the query plan before distributed execution begins.

Row processing happens later, inside tasks, when Spark actually executes the plan across partitions. Do not describe this as “Catalyst compiling across millions of rows.” Catalyst plans the work; tasks process the rows.

### ⏱️ Performance & Validation Summary

* **Step 1:** Started a local Spark session and printed the actual Spark UI URL.
* **Step 2:** Fabricated an asymmetric key distribution to mimic skew.
* **Step 3:** Routed skewed data through a shuffle boundary to inspect stage and task metrics.
* **Step 4:** Kept the live UI open long enough to audit task duration, shuffle read, spill, and GC signals.